Objective:
=============
The purpose of this python notebook is to take the combined Dataset (Factory_Data_By_District.csv), and remove/ Filter the unnecessary columns and rows according to SNRE Specifications.

1. Only interested in leasehold tenures (of industrial properties) b/c majority of transaction are related to this type of property.
2. Only Interested in Multiple User Factory Property found in Strata
3. Clean the nesessary dataset from singstats.gov.sg and merge with the Factory dataset
4. Add the lagged psf value - an common industry practice
5. Add the Rental Index for Multiple User Factory - Based on Region (Central, East, North, NE, West) - 28 Feb 2024




In [1]:
import pandas as pd
pd.options.mode.chained_assignment = None # Disable warning from pandas library
# pd.options.mode.chained_assignment = 'warn' # enable warning from pandas library
from datetime import datetime
import re

In [2]:
#Custom function to remove metadata
def removeMeta(dfName):
    firstColName = dfName.columns[0]
    startIndex, endIndex = 0,0
    startIndex = dfName.index[dfName[firstColName]=='Data Series'].tolist()[0]
    endIndex = dfName.index[dfName[firstColName]=='Footnotes:'].tolist()[0]-1
    dfName = dfName.iloc[startIndex:endIndex] #remove metadata
    return dfName

def checkNull(dfName):
    null_counts = dfName.isnull().sum()
    null_counts = pd.Series(null_counts, name = 'Null Count').to_frame()
    print("Rows: {}, Cols: {}".format(dfName.shape[0],dfName.shape[1]))
    print('')
    print(dfName.info())
    print('')
    display(null_counts[null_counts['Null Count'] > 0])
    

In [3]:
# Read Factory_Data_By_District.csv file and load into the dataframe, factoryDF
# Remember that Factory_Data_By_District is simply the compilation of all the transaction (by postal district) csvs.
factoryDF = pd.read_csv('Data/Factory_Data_By_District.csv', na_values=['NA', 'N/A','na','-']) 

checkNull(factoryDF)
display(factoryDF.tail(5))

Rows: 38241, Cols: 15

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38241 entries, 0 to 38240
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Contract Date       38241 non-null  object
 1   Project Name        38091 non-null  object
 2   Street Name         38241 non-null  object
 3   Property Type       38241 non-null  object
 4   Type Of Area        38241 non-null  object
 5   Area (sqft)         38241 non-null  object
 6   Floor Level         37132 non-null  object
 7   Price               38241 non-null  object
 8   Unit Price ($ psf)  38241 non-null  object
 9   Tenure              38234 non-null  object
 10  Type of Sale        38241 non-null  object
 11  Region              38241 non-null  object
 12  Planning Area       38241 non-null  object
 13  Postal District     38241 non-null  int64 
 14  Postal Sector       38241 non-null  int64 
dtypes: int64(2), object(13)
memory usage: 4.4+ MB
N

,Null Count
Project Name,150
Floor Level,1109
Tenure,7


,Contract Date,Project Name,Street Name,Property Type,Type Of Area,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector
38236,12/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,Multiple-User Factory,Strata,"1,701",NaN,"585,561",344,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75
38237,12/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,Multiple-User Factory,Strata,"1,173",NaN,"375,448",320,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75
38238,19/3/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,Multiple-User Factory,Strata,"13,519",NaN,"3,900,000",288,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75
38239,12/2/1996,ADMIRALTY INDUSTRIAL PARK,ADMIRALTY ROAD WEST,Multiple-User Factory,Strata,"22,755",NaN,"5,803,290",255,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75
38240,12/1/1996,ADMIRALTY INDUSTRIAL PARK,ADMIRALTY ROAD WEST,Multiple-User Factory,Strata,"1,561",NaN,"846,588",542,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75


In [4]:
# Narrow down our dataset into our specific use case - leasehold, strata, Multiple-User Factory Properties.
# Note: Leasehold with 999 years is considered freehold
# Note: Narrow down because majority of their transaction is leasehold
# Note: I only want 'Strata' and Non-'Freehold' properties

# Using Boolean masking to remove 'Freehold' and non-'Strata' first
tenureMask = factoryDF['Tenure'] !='Freehold'
areaTypeMask = factoryDF['Type Of Area'] == 'Strata'
leaseholdFactoryDF = factoryDF[tenureMask & areaTypeMask].reset_index(drop=True)

# Using Boolean masking to remove 'Tenure' containing '999 yrs'
longLeasePattern = r'999 yrs'
longLeaseMask = leaseholdFactoryDF['Tenure'].str.contains(longLeasePattern, case=False, na=False)
leaseholdFactoryDF = leaseholdFactoryDF[~longLeaseMask].reset_index(drop=True)

checkNull(leaseholdFactoryDF)

Rows: 32150, Cols: 15

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32150 entries, 0 to 32149
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Contract Date       32150 non-null  object
 1   Project Name        32040 non-null  object
 2   Street Name         32150 non-null  object
 3   Property Type       32150 non-null  object
 4   Type Of Area        32150 non-null  object
 5   Area (sqft)         32150 non-null  object
 6   Floor Level         31159 non-null  object
 7   Price               32150 non-null  object
 8   Unit Price ($ psf)  32150 non-null  object
 9   Tenure              32143 non-null  object
 10  Type of Sale        32150 non-null  object
 11  Region              32150 non-null  object
 12  Planning Area       32150 non-null  object
 13  Postal District     32150 non-null  int64 
 14  Postal Sector       32150 non-null  int64 
dtypes: int64(2), object(13)
memory usage: 3.7+ MB
N

,Null Count
Project Name,110
Floor Level,991
Tenure,7


In [5]:
# Remove records with NA values (e.g. - ) in ['Tenure', 'Floor Level'] or invalid data (e.g Tenure with no start Year)
# Tenure and Floor level is important and cannot anyhow impute. Based on Clarence, these two are significant variables.
# Question: should i be removing if tenure has no start year? because I can't get the remaining tenure.

#Using Regex and Boolean Masking
leaseholdFactoryDF = leaseholdFactoryDF.dropna(subset=['Tenure', 'Floor Level']) 
noDatePattern = r'yrs from \d{2}\/\d{2}\/\d{4}'
noDateMask = leaseholdFactoryDF['Tenure'].str.contains(noDatePattern, case=False, na=False)
leaseholdFactoryDF = leaseholdFactoryDF[noDateMask]
leaseholdFactoryDF = leaseholdFactoryDF.reset_index(drop=True) #reset the index, just in case.

checkNull(leaseholdFactoryDF)

Rows: 31145, Cols: 15

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31145 entries, 0 to 31144
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Contract Date       31145 non-null  object
 1   Project Name        31064 non-null  object
 2   Street Name         31145 non-null  object
 3   Property Type       31145 non-null  object
 4   Type Of Area        31145 non-null  object
 5   Area (sqft)         31145 non-null  object
 6   Floor Level         31145 non-null  object
 7   Price               31145 non-null  object
 8   Unit Price ($ psf)  31145 non-null  object
 9   Tenure              31145 non-null  object
 10  Type of Sale        31145 non-null  object
 11  Region              31145 non-null  object
 12  Planning Area       31145 non-null  object
 13  Postal District     31145 non-null  int64 
 14  Postal Sector       31145 non-null  int64 
dtypes: int64(2), object(13)
memory usage: 3.6+ MB
N

,Null Count
Project Name,81


In [6]:
#To extract the Quarter, month & year from "Contract Date" Column for mapping purposes
#Note: Quarter is in string datatype, Month and Year is int datatype

quarterList,monthList ,yearList = [], [], []

for i in leaseholdFactoryDF['Contract Date']:
    month = int(i.split("/")[1])
    year = int(i.split("/")[2])
    monthinquarter = str((month-1)//3+1)
    quarter = str(year) + " " + monthinquarter +"Q"  
    quarterList.append(quarter)
    monthList.append(month)
    yearList.append(year)

#Convert list into pd Series    
quarterSeries = pd.Series(quarterList, name='Quarter')
monthSeries = pd.Series(monthList, name='Month')
yearSeries = pd.Series(yearList, name='Year')

# Add quarterSeries, monthSeries & yearSeries as new columns into leaseholdFactoryDF
leaseholdFactoryDF['Quarter'] = quarterSeries
leaseholdFactoryDF['Month'] = monthSeries
leaseholdFactoryDF['Year'] = yearSeries

# print(leaseholdFactoryDF.shape)
checkNull(leaseholdFactoryDF)
display(leaseholdFactoryDF.tail(3))


Rows: 31145, Cols: 18

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31145 entries, 0 to 31144
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Contract Date       31145 non-null  object
 1   Project Name        31064 non-null  object
 2   Street Name         31145 non-null  object
 3   Property Type       31145 non-null  object
 4   Type Of Area        31145 non-null  object
 5   Area (sqft)         31145 non-null  object
 6   Floor Level         31145 non-null  object
 7   Price               31145 non-null  object
 8   Unit Price ($ psf)  31145 non-null  object
 9   Tenure              31145 non-null  object
 10  Type of Sale        31145 non-null  object
 11  Region              31145 non-null  object
 12  Planning Area       31145 non-null  object
 13  Postal District     31145 non-null  int64 
 14  Postal Sector       31145 non-null  int64 
 15  Quarter             31145 non-null  object
 16 

,Null Count
Project Name,81


,Contract Date,Project Name,Street Name,Property Type,Type Of Area,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year
31142,13/12/1996,ADMIRALTY INDUSTRIAL PARK,ADMIRALTY ROAD WEST,Multiple-User Factory,Strata,"1,636",Non-First Floor,"638,089",390,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 4Q,12,1996
31143,16/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,Multiple-User Factory,Strata,"1,625",Non-First Floor,"560,750",345,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 2Q,4,1996
31144,15/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,Multiple-User Factory,Strata,"1,184",Non-First Floor,"374,318",316,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 2Q,4,1996


In [7]:
#import pandas as pd

# # Sample dates
# dates_series = leaseholdFactoryDF['Contract Date']

# # Convert the Series of strings to datetime objects
# dates_series = pd.to_datetime(dates_series, dayfirst = True)

# display(dates_series.head(105))

# # Lagged year
# lagged_year = dates_series.dt.year - 1

# # Lagged month with year
# lagged_month_year = dates_series - pd.DateOffset(months=1)

# display(lagged_month_year.head(105))

# Lagged quarter with year
# lagged_quarter_year = dates_series - pd.DateOffset(months=3)

# display(lagged_quarter_year.head(105))

# # Print the results
# print("Original Date:", dates_series)
# print("Lagged Year:", lagged_year)
# print("Lagged Month with Year:", lagged_month_year)
# print("Lagged Quarter with Year:", lagged_quarter_year)
          

In [8]:
# From the 'Tenure Column', find the remaining tenure duration when the transaction was made.
# take leaseholdFactoryDF.iloc[0,:], where contract year = 2023 and tenure is 99 years from 1962
# therefore, (1962 + 99) - 2023 = 38 years left at the time of transaction.

endTenureList = []
for i in leaseholdFactoryDF['Tenure']:
    tenureYears = int(i.split(" ")[0])

    if re.search(r'/', i.split(" ")[-1]): # check for 'ETC' in the 'Tenure' Column
        startTenure = int(i.split(" ")[-1].split("/")[-1])
    else:
        startTenure = int(i.split(" ")[-2].split("/")[-1])    
    endTenure = tenureYears + startTenure
    endTenureList.append(endTenure)

endTenureSeries = pd.Series(endTenureList, name='endTenure')
leaseholdFactoryDF['Remaining Tenure'] = endTenureSeries - leaseholdFactoryDF['Year']
leaseholdFactoryDF.reset_index(drop=True,inplace=True)

checkNull(leaseholdFactoryDF)
display(type(leaseholdFactoryDF['Month'][0]))

display(leaseholdFactoryDF.head(5))


Rows: 31145, Cols: 19

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31145 entries, 0 to 31144
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Contract Date       31145 non-null  object
 1   Project Name        31064 non-null  object
 2   Street Name         31145 non-null  object
 3   Property Type       31145 non-null  object
 4   Type Of Area        31145 non-null  object
 5   Area (sqft)         31145 non-null  object
 6   Floor Level         31145 non-null  object
 7   Price               31145 non-null  object
 8   Unit Price ($ psf)  31145 non-null  object
 9   Tenure              31145 non-null  object
 10  Type of Sale        31145 non-null  object
 11  Region              31145 non-null  object
 12  Planning Area       31145 non-null  object
 13  Postal District     31145 non-null  int64 
 14  Postal Sector       31145 non-null  int64 
 15  Quarter             31145 non-null  object
 16 

,Null Count
Project Name,81


numpy.int64

,Contract Date,Project Name,Street Name,Property Type,Type Of Area,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure
0,25/5/2023,E-CENTRE @ REDHILL,JALAN BUKIT MERAH,Multiple-User Factory,Strata,969,Non-First Floor,"700,000",722,99 yrs from 01/07/1962,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,38
1,16/5/2023,REDHILL FORUM,JALAN KILANG TIMOR,Multiple-User Factory,Strata,969,Non-First Floor,"489,800",505,99 yrs from 01/01/1961,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,37
2,3/5/2023,KEWALRAM HOUSE,JALAN KILANG TIMOR,Multiple-User Factory,Strata,"4,295",Non-First Floor,"1,700,000",396,99 yrs from 01/01/1961,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,37
3,27/4/2023,E-CENTRE @ REDHILL,JALAN BUKIT MERAH,Multiple-User Factory,Strata,"1,012",Non-First Floor,"600,000",593,99 yrs from 01/07/1962,Resale,Central Region,Bukit Merah,3,15,2023 2Q,4,2023,38
4,28/3/2023,E-CENTRE @ REDHILL,JALAN BUKIT MERAH,Multiple-User Factory,Strata,"1,012",Non-First Floor,"593,700",587,99 yrs from 01/07/1962,Resale,Central Region,Bukit Merah,3,15,2023 1Q,3,2023,38


Explanation:
------------------------
The following steps is used to transform the economic factors (from their respective .csv files) and map them back to the leaseholdFactoryDF

In [9]:
#Process the Population data (from singstats)
#Final results is a dataframe, sgPOPDF, with the Population figures based on each year.

sgPOPDF = pd.read_csv('Data/2024/population.csv') 
headerStart, headerEnd = 0,0

sgPOPDF = removeMeta(sgPOPDF)
sgPOPDF = sgPOPDF.set_index('Unnamed: 0')

headerStart = int(sgPOPDF.loc['Data Series'][0])
headerEnd = int(sgPOPDF.loc['Data Series'][-1])

sgPOPDF.columns = pd.Series(range(headerStart, headerEnd-1, -1))

sgPOPDF = sgPOPDF.iloc[1,0:] #is now a series
sgPOPDF = sgPOPDF.reset_index()
sgPOPDF = sgPOPDF.rename(columns={'index': 'Year'})

sgPOPDF = sgPOPDF.astype({'Year': 'int', 
                          'Total Population (Number)': 'float64'})

display(type(sgPOPDF['Total Population (Number)'][0]))
display(sgPOPDF)


numpy.float64

,Year,Total Population (Number)
0,2023,5917648.0
1,2022,5637022.0
2,2021,5453566.0
3,2020,5685807.0
4,2019,5703569.0
...,...,...
69,1954,1248200.0
70,1953,1191800.0
71,1952,1127000.0
72,1951,1068100.0


In [10]:
#Process the Sora data (from singstats)
#Final results is a dataframe, soraDF, with the SORA figures based on each month of the year.

soraDF = pd.read_csv('Data/2024/SORA.csv',na_values=['NA', 'N/A','na'])
soraYear, soraMonth = [],[]
soraDF = removeMeta(soraDF)
soraDF = soraDF.iloc[[0,-1],:].T.reset_index(drop=True)
soraDF.columns = soraDF.iloc[0]
soraDF = soraDF.iloc[1:].reset_index(drop=True)
soraDF = soraDF.rename(columns={'Data Series': 'Quarter', 'Singapore Overnight Rate Average':'SORA'})


#convert the "Quarter" column to date object so we can translate the month to number. 
#so we can map back to transactional dataset
soraDF['Quarter'] = pd.to_datetime(soraDF['Quarter'])

for i in soraDF['Quarter']:
    soraYear.append(i.year)
    soraMonth.append(i.month)
    
soraYearSeries = pd.Series(soraYear)
soraMonthSeries = pd.Series(soraMonth)
soraDF['Year'] = soraYearSeries 
soraDF['Month'] = soraMonthSeries

#Remove rows with na SORA Values and reset index
soraDF = soraDF.dropna(subset=['SORA'])
soraDF = soraDF.reset_index(drop=True)

display(type(soraDF['Month'][0]))
display(type(soraDF['Year'][0]))
display(soraDF.head(3))

numpy.int64

numpy.int64

,Quarter,SORA,Year,Month
0,2023-12-01,3.6235,2023,12
1,2023-11-01,3.6592,2023,11
2,2023-10-01,3.6904,2023,10


In [11]:
#Process the GDP data (from singstats)

yearlyNominalGDPDF = pd.read_csv('Data/2024/Annual_GDP.csv')
deflatorDF = pd.read_csv('Data/2024/Annual_GDP_Deflator.csv')
yearlyNominalGDPDF = removeMeta(yearlyNominalGDPDF)
deflatorDF = removeMeta(deflatorDF)

yearlyNominalGDPDF = yearlyNominalGDPDF.iloc[[0,1],:].T.reset_index(drop=True)
yearlyNominalGDPDF.columns = yearlyNominalGDPDF.iloc[0]
yearlyNominalGDPDF = yearlyNominalGDPDF.iloc[1:].reset_index(drop=True)
yearlyNominalGDPDF = yearlyNominalGDPDF.rename(columns={'Data Series': 'Year', 
                              'GDP At Current Market Prices':'Nominal Annual GDP (in $Sm)'})


deflatorDF = deflatorDF.iloc[[0,1],:].T.reset_index(drop=True)
deflatorDF.columns = deflatorDF.iloc[0]
deflatorDF = deflatorDF.iloc[1:].reset_index(drop=True)
deflatorDF = deflatorDF.rename(columns={'Data Series': 'Year', 'GDP':'GDP Deflator'})

gdpDF = deflatorDF.merge(yearlyNominalGDPDF, on='Year')

gdpDF = gdpDF.astype({'Year': 'int', 
                      'GDP Deflator': 'float64', 
                      'Nominal Annual GDP (in $Sm)': 'float64'})

#Change Deflator to decimal point
gdpDF['GDP Deflator'] = gdpDF['GDP Deflator']/100

# Calculate Real GDP
gdpDF['Real GDP (in $Sm)'] = (gdpDF['Nominal Annual GDP (in $Sm)'] / gdpDF['GDP Deflator'])
gdpDF['Real GDP (in $Sm)'] = gdpDF['Real GDP (in $Sm)'].round(2)

display(gdpDF)

,Year,GDP Deflator,Nominal Annual GDP (in $Sm),Real GDP (in $Sm)
0,2022,1.233,643545.8,521934.96
1,2021,1.130,569364.2,503862.12
2,2020,1.039,480691.2,462647.93
3,2019,1.068,514066.0,481335.21
4,2018,1.070,508337.4,475081.68
...,...,...,...,...
58,1964,0.270,2737.2,10137.78
59,1963,0.268,2809.0,10481.34
60,1962,0.266,2529.3,9508.65
61,1961,0.265,2340.7,8832.83


In [12]:
#Process the land supply data (from singstats) - Measured in square metre
#Final results is a dataframe, landSupplyDF, with the amount of industrial space of the year.
landSupplyDF = pd.read_csv('Data/2024/Supply Of Commercial And Industrial Properties In The Pipeline By Development Status.csv', na_values=['NA', 'N/A','na'])
landSupplyDF = removeMeta(landSupplyDF)
landSupplyFirstColName = landSupplyDF.columns[0]

landSupplyQuarterIndex = landSupplyDF.index[landSupplyDF[landSupplyFirstColName]=='Data Series'].tolist()[0]
warehouseIndex = landSupplyDF.index[landSupplyDF[landSupplyFirstColName]=='Total Warehouse Space'].tolist()[0]
singleFactoryIndex = landSupplyDF.index[landSupplyDF[landSupplyFirstColName]=='Total Single-User Factory Space'].tolist()[0]
multiFactoryIndex = landSupplyDF.index[landSupplyDF[landSupplyFirstColName]=='Total Multiple-User Factory Space'].tolist()[0]
bizParkIndex = landSupplyDF.index[landSupplyDF[landSupplyFirstColName]=='Total Business Park Space'].tolist()[0]

landSupplyDF = landSupplyDF.loc[[landSupplyQuarterIndex, warehouseIndex, singleFactoryIndex, multiFactoryIndex,bizParkIndex],:].T.reset_index(drop=True)
landSupplyDF.columns = landSupplyDF.iloc[0]
landSupplyDF = landSupplyDF.iloc[1:].reset_index(drop=True)
landSupplyDF = landSupplyDF.rename(columns={'Data Series': 'Quarter'})

#Remove rows with na Values and reset index
landSupplyDF = landSupplyDF.dropna(subset=['Total Warehouse Space','Total Single-User Factory Space',
                                           'Total Multiple-User Factory Space','Total Business Park Space'])
landSupplyDF = landSupplyDF.reset_index(drop=True)

landSupplyDF = landSupplyDF.astype({'Quarter': 'str', 
                                    'Total Warehouse Space': 'float64', 
                                    'Total Single-User Factory Space': 'float64',
                                    'Total Multiple-User Factory Space': 'float64', 
                                    'Total Business Park Space': 'float64'})

#remove ghost spaces so that we can merge the DFs
landSupplyDF['Quarter'] = landSupplyDF['Quarter'].apply(lambda x: x.strip())

landSupplyDF['Total Warehouse Space'] = landSupplyDF['Total Warehouse Space'] * 1000
landSupplyDF['Total Single-User Factory Space'] = landSupplyDF['Total Single-User Factory Space'] * 1000
landSupplyDF['Total Multiple-User Factory Space'] = landSupplyDF['Total Multiple-User Factory Space'] * 1000
landSupplyDF['Total Business Park Space'] = landSupplyDF['Total Business Park Space'] * 1000

landSupplyDF['Total Supply of Industrial Space (m2)'] = landSupplyDF['Total Warehouse Space'] + landSupplyDF['Total Single-User Factory Space'] + landSupplyDF['Total Multiple-User Factory Space'] + landSupplyDF['Total Business Park Space'] 

display(type(landSupplyDF['Quarter'][0]))
display(landSupplyDF.head(3))


str

,Quarter,Total Warehouse Space,Total Single-User Factory Space,Total Multiple-User Factory Space,Total Business Park Space,Total Supply of Industrial Space (m2)
0,2023 4Q,903000.0,1484000.0,833000.0,347000.0,3567000.0
1,2023 3Q,885000.0,1527000.0,672000.0,411000.0,3495000.0
2,2023 2Q,631000.0,1795000.0,766000.0,420000.0,3612000.0


In [13]:
#Process the average yearly Forex data (from singstats)
forexDF = pd.read_csv('Data/2024/Exchange Rates (Average For The Year).csv', na_values=['NA', 'N/A','na'])
forexYear, forexMonth = [],[]
forexDF = removeMeta(forexDF)
forexFirstColName = forexDF.columns[0]

forexYearIndex = forexDF.index[forexDF[forexFirstColName]=='Data Series'].tolist()[0]
usdIndex = forexDF.index[forexDF[forexFirstColName]=='US Dollar (Singapore Dollar Per US Dollar)'].tolist()[0]

forexDF = forexDF.loc[[forexYearIndex,usdIndex],:].T.reset_index(drop=True)
forexDF.columns = forexDF.iloc[0]
forexDF = forexDF.iloc[1:].reset_index(drop=True)
forexDF = forexDF.rename(columns={'Data Series': 'Year'})

forexDF = forexDF.astype({'Year': 'int',
                          'US Dollar (Singapore Dollar Per US Dollar)': 'float64'})

display(forexDF)

,Year,US Dollar (Singapore Dollar Per US Dollar)
0,2023,1.3431
1,2022,1.3789
2,2021,1.3440
3,2020,1.3792
4,2019,1.3642
5,2018,1.3491
6,2017,1.3807
7,2016,1.3815
8,2015,1.3748
9,2014,1.2671


In [14]:
# Data about the Tender Price Index is manually extract from the below sources
# https://www1.bca.gov.sg/docs/default-source/docs-corp-form/free-stats.pdf
# https://www.sisv.org.sg/admin/efinder/files/SISV%20TPI_3Q2023.pdf
# https://surbanajurong.com/wp-content/uploads/download/SGMarketReviewOutlook2023-20230518.pdf

bcaTPIDF = pd.DataFrame({'Year':[2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023],
                         'TPI' :[119.1, 101.3, 100.0, 99.7, 99.8, 104.6, 106.8, 104.0, 98.0, 96.7, 98.6, 99.9, 102.8,
                                 117.1, 130.7, 136.0]})

bcaTPIDF['TPI % Change'] = bcaTPIDF['TPI'] - 100

bcaTPIDF = bcaTPIDF.astype({'Year': 'int', 
                            'TPI': 'float64', 
                            'TPI % Change': 'float64'})

display(bcaTPIDF)

,Year,TPI,TPI % Change
0,2008,119.1,19.1
1,2009,101.3,1.3
2,2010,100.0,0.0
3,2011,99.7,-0.3
4,2012,99.8,-0.2
5,2013,104.6,4.6
6,2014,106.8,6.8
7,2015,104.0,4.0
8,2016,98.0,-2.0
9,2017,96.7,-3.3


In [15]:
#Now we merge all the DF into one final dataset

print("Leasehold DF: {}\n".format(leaseholdFactoryDF.shape))
print("Population DF: {}\n{}\n".format(sgPOPDF.shape, sgPOPDF.columns))
print("SORA DF: {}\n{}\n".format(soraDF.shape, soraDF.columns))
print("GDP DF: {}\n{}\n".format(gdpDF.shape, gdpDF.columns))
print("LandSupply DF: {}\n{}\n".format(landSupplyDF.shape, landSupplyDF.columns))
print("forex DF: {}\n{}\n".format(forexDF.shape, forexDF.columns))
print("bcaTPI DF: {}\n{}\n".format(bcaTPIDF.shape, bcaTPIDF.columns))



Leasehold DF: (31145, 19)

Population DF: (74, 2)
Index(['Year', 'Total Population (Number)'], dtype='object')

SORA DF: (222, 4)
Index(['Quarter', 'SORA', 'Year', 'Month'], dtype='object', name=0)

GDP DF: (63, 4)
Index(['Year', 'GDP Deflator', 'Nominal Annual GDP (in $Sm)',
       'Real GDP (in $Sm)'],
      dtype='object', name=0)

LandSupply DF: (41, 6)
Index(['Quarter', 'Total Warehouse Space', 'Total Single-User Factory Space',
       'Total Multiple-User Factory Space', 'Total Business Park Space',
       'Total Supply of Industrial Space (m2)'],
      dtype='object', name=0)

forex DF: (36, 2)
Index(['Year', 'US Dollar (Singapore Dollar Per US Dollar)'], dtype='object', name=0)

bcaTPI DF: (16, 3)
Index(['Year', 'TPI', 'TPI % Change'], dtype='object')



In [16]:
#Drop unnecessary columns
leaseholdFactoryDF = leaseholdFactoryDF.drop(columns=['Property Type','Type Of Area'])
display(leaseholdFactoryDF.tail(3))
display(sgPOPDF.tail(3))


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure
31142,13/12/1996,ADMIRALTY INDUSTRIAL PARK,ADMIRALTY ROAD WEST,"1,636",Non-First Floor,"638,089",390,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 4Q,12,1996,59
31143,16/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,"1,625",Non-First Floor,"560,750",345,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 2Q,4,1996,59
31144,15/4/1996,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,"1,184",Non-First Floor,"374,318",316,60 yrs from 09/01/1995,New Sale,North Region,Woodlands,27,75,1996 2Q,4,1996,59


,Year,Total Population (Number)
71,1952,1127000.0
72,1951,1068100.0
73,1950,1022100.0


In [17]:
#Merge xsaction with population
finalDF1 = leaseholdFactoryDF.merge(sgPOPDF,how='inner', on='Year')
print(finalDF1.shape)
display(finalDF1.tail(3))


(31145, 18)


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure,Total Population (Number)
31142,1/4/1997,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,"1,808",Non-First Floor,"709,455",392,60 yrs from 09/01/1995,Subsale,North Region,Woodlands,27,75,1997 2Q,4,1997,58,3796038.0
31143,23/1/1997,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,"1,173",Non-First Floor,"481,043",410,60 yrs from 09/01/1995,Subsale,North Region,Woodlands,27,75,1997 1Q,1,1997,58,3796038.0
31144,13/1/1997,ADMIRALTY INDUSTRIAL PARK,WOODLANDS INDUSTRIAL PARK E1,"1,173",Non-First Floor,"480,000",409,60 yrs from 09/01/1995,Subsale,North Region,Woodlands,27,75,1997 1Q,1,1997,58,3796038.0


In [18]:
soraDF = soraDF[['Month','Year','SORA']]
display(soraDF.tail(3))


,Month,Year,SORA
219,9,2005,2.2293
220,8,2005,1.9718
221,7,2005,1.7712


In [19]:
#Merge with finalDF1 with soraDF to create new DF, finalDF2
finalDF2 = finalDF1.merge(soraDF,how='inner', on=['Month','Year'])
print(finalDF2.shape)
display(finalDF2.head(3))


(26821, 19)


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA
0,25/5/2023,E-CENTRE @ REDHILL,JALAN BUKIT MERAH,969,Non-First Floor,"700,000",722,99 yrs from 01/07/1962,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,38,5917648.0,3.7241
1,16/5/2023,REDHILL FORUM,JALAN KILANG TIMOR,969,Non-First Floor,"489,800",505,99 yrs from 01/01/1961,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,37,5917648.0,3.7241
2,3/5/2023,KEWALRAM HOUSE,JALAN KILANG TIMOR,"4,295",Non-First Floor,"1,700,000",396,99 yrs from 01/01/1961,Resale,Central Region,Bukit Merah,3,15,2023 2Q,5,2023,37,5917648.0,3.7241


In [20]:
gdpDF = gdpDF[['Year', 'Real GDP (in $Sm)']]
display(gdpDF.head(3))


,Year,Real GDP (in $Sm)
0,2022,521934.96
1,2021,503862.12
2,2020,462647.93


In [21]:
#Merge with finalDF2 with gdpDF to create new DF, finalDF3 based on the 'Year' column
finalDF3 = finalDF2.merge(gdpDF,how='inner', on='Year')
print(finalDF3.shape)
display(finalDF3)

(26348, 20)


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA,Real GDP (in $Sm)
0,27/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"2,852",Non-First Floor,"1,070,000",375,30 yrs from 01/03/2008,Resale,Central Region,Queenstown,3,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96
1,20/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"1,410",Non-First Floor,"430,000",305,30 yrs from 01/03/2008,Resale,Central Region,Queenstown,3,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96
2,22/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"5,350",First Floor,"3,157,000",590,99 yrs from 27/01/1984,Resale,West Region,Clementi,5,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96
3,8/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"1,109",First Floor,"621,040",560,99 yrs from 27/01/1984,Resale,West Region,Clementi,5,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96
4,28/12/2022,CT HUB 2,LAVENDER STREET,"1,152",Non-First Floor,"1,050,000",911,63 yrs from 12/01/2012,Resale,Central Region,Kallang,12,33,2022 4Q,12,2022,53,5637022.0,2.5293,521934.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26343,8/7/2005,NORTH LINK BUILDING,ADMIRALTY STREET,"5,188",Non-First Floor,"628,000",121,60 yrs from 09/10/1999,Resale,North Region,Sembawang,27,75,2005 3Q,7,2005,54,4265762.0,1.7712,245072.58
26344,8/7/2005,NORTH LINK BUILDING,ADMIRALTY STREET,"5,188",Non-First Floor,"628,000",121,60 yrs from 09/10/1999,Resale,North Region,Sembawang,27,75,2005 3Q,7,2005,54,4265762.0,1.7712,245072.58
26345,7/7/2005,NORTH LINK BUILDING,ADMIRALTY STREET,"10,376",Non-First Floor,"1,256,000",121,60 yrs from 09/10/1999,Resale,North Region,Sembawang,27,75,2005 3Q,7,2005,54,4265762.0,1.7712,245072.58
26346,5/7/2005,NORTH LINK BUILDING,ADMIRALTY STREET,"5,188",Non-First Floor,"628,000",121,60 yrs from 09/10/1999,Resale,North Region,Sembawang,27,75,2005 3Q,7,2005,54,4265762.0,1.7712,245072.58


In [22]:
# Since my target property population is Multiple-User Factory, i should only be interested in this type of property.
landSupplyDF = landSupplyDF[['Quarter', 'Total Multiple-User Factory Space']]
display(landSupplyDF.tail(3))


,Quarter,Total Multiple-User Factory Space
38,2014 2Q,2360000.0
39,2014 1Q,2268000.0
40,2013 4Q,2230000.0


In [23]:
#Merge with finalDF3 with landSupplyDF to create new DF, finalDF4
finalDF4 = finalDF3.merge(landSupplyDF,how='inner', on='Quarter')
print(finalDF4.shape)
display(finalDF4)

(10010, 21)


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,...,Postal District,Postal Sector,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA,Real GDP (in $Sm),Total Multiple-User Factory Space
0,27/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"2,852",Non-First Floor,"1,070,000",375,30 yrs from 01/03/2008,Resale,Central Region,...,3,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0
1,20/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"1,410",Non-First Floor,"430,000",305,30 yrs from 01/03/2008,Resale,Central Region,...,3,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0
2,22/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"5,350",First Floor,"3,157,000",590,99 yrs from 27/01/1984,Resale,West Region,...,5,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0
3,8/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"1,109",First Floor,"621,040",560,99 yrs from 27/01/1984,Resale,West Region,...,5,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0
4,28/12/2022,CT HUB 2,LAVENDER STREET,"1,152",Non-First Floor,"1,050,000",911,63 yrs from 12/01/2012,Resale,Central Region,...,12,33,2022 4Q,12,2022,53,5637022.0,2.5293,521934.96,790000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10005,5/12/2013,HARVEST @ WOODLANDS,WOODLANDS INDUSTRIAL PARK E5,"1,690",Non-First Floor,"590,000",349,60 yrs from 05/10/2009,Resale,North Region,...,27,75,2013 4Q,12,2013,56,5399162.0,0.0698,395550.15,2230000.0
10006,4/12/2013,NORTH POINT BIZHUB,YISHUN INDUSTRIAL STREET 1,"2,562",Non-First Floor,"750,000",293,57 yrs from 18/09/2012,Resale,North Region,...,27,76,2013 4Q,12,2013,56,5399162.0,0.0698,395550.15,2230000.0
10007,3/12/2013,WIN 5,YISHUN INDUSTRIAL STREET 1,"2,895",Non-First Floor,"626,393",216,30 yrs from 05/12/2012,New Sale,North Region,...,27,76,2013 4Q,12,2013,29,5399162.0,0.0698,395550.15,2230000.0
10008,3/12/2013,NORTH SPRING BIZHUB,YISHUN INDUSTRIAL STREET 1,"1,593",Non-First Floor,"700,000",439,60 yrs from 01/02/2011,Subsale,North Region,...,27,76,2013 4Q,12,2013,58,5399162.0,0.0698,395550.15,2230000.0


In [24]:
forexDF = forexDF[['Year','US Dollar (Singapore Dollar Per US Dollar)']]
display(forexDF.tail(3))


,Year,US Dollar (Singapore Dollar Per US Dollar)
33,1990,1.8125
34,1989,1.9503
35,1988,2.0124


In [25]:
#Merge with finalDF4 with landSupplyDF to create new DF, finalDF5
finalDF5 = finalDF4.merge(forexDF,how='inner', on=['Year'])
print(finalDF5.shape)
display(finalDF5)

(10010, 22)


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,...,Postal Sector,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA,Real GDP (in $Sm),Total Multiple-User Factory Space,US Dollar (Singapore Dollar Per US Dollar)
0,27/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"2,852",Non-First Floor,"1,070,000",375,30 yrs from 01/03/2008,Resale,Central Region,...,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789
1,20/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"1,410",Non-First Floor,"430,000",305,30 yrs from 01/03/2008,Resale,Central Region,...,14,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789
2,22/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"5,350",First Floor,"3,157,000",590,99 yrs from 27/01/1984,Resale,West Region,...,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0,1.3789
3,8/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"1,109",First Floor,"621,040",560,99 yrs from 27/01/1984,Resale,West Region,...,12,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0,1.3789
4,28/12/2022,CT HUB 2,LAVENDER STREET,"1,152",Non-First Floor,"1,050,000",911,63 yrs from 12/01/2012,Resale,Central Region,...,33,2022 4Q,12,2022,53,5637022.0,2.5293,521934.96,790000.0,1.3789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10005,5/12/2013,HARVEST @ WOODLANDS,WOODLANDS INDUSTRIAL PARK E5,"1,690",Non-First Floor,"590,000",349,60 yrs from 05/10/2009,Resale,North Region,...,75,2013 4Q,12,2013,56,5399162.0,0.0698,395550.15,2230000.0,1.2513
10006,4/12/2013,NORTH POINT BIZHUB,YISHUN INDUSTRIAL STREET 1,"2,562",Non-First Floor,"750,000",293,57 yrs from 18/09/2012,Resale,North Region,...,76,2013 4Q,12,2013,56,5399162.0,0.0698,395550.15,2230000.0,1.2513
10007,3/12/2013,WIN 5,YISHUN INDUSTRIAL STREET 1,"2,895",Non-First Floor,"626,393",216,30 yrs from 05/12/2012,New Sale,North Region,...,76,2013 4Q,12,2013,29,5399162.0,0.0698,395550.15,2230000.0,1.2513
10008,3/12/2013,NORTH SPRING BIZHUB,YISHUN INDUSTRIAL STREET 1,"1,593",Non-First Floor,"700,000",439,60 yrs from 01/02/2011,Subsale,North Region,...,76,2013 4Q,12,2013,58,5399162.0,0.0698,395550.15,2230000.0,1.2513


In [26]:
bcaTPIDF = bcaTPIDF[['Year','TPI % Change']]
display(bcaTPIDF.tail(3))

,Year,TPI % Change
13,2021,17.1
14,2022,30.7
15,2023,36.0


In [27]:
#Merge with finalDF5 with bcaTPIDF to create new DF, finalDF
finalDF = finalDF5.merge(bcaTPIDF,how='inner', on=['Year'])
print(finalDF.shape)
print(finalDF.columns)
display(finalDF.head(3))

(10010, 23)
Index(['Contract Date', 'Project Name', 'Street Name', 'Area (sqft)',
       'Floor Level', 'Price', 'Unit Price ($ psf)', 'Tenure', 'Type of Sale',
       'Region', 'Planning Area', 'Postal District', 'Postal Sector',
       'Quarter', 'Month', 'Year', 'Remaining Tenure',
       'Total Population (Number)', 'SORA', 'Real GDP (in $Sm)',
       'Total Multiple-User Factory Space',
       'US Dollar (Singapore Dollar Per US Dollar)', 'TPI % Change'],
      dtype='object')


,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,...,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA,Real GDP (in $Sm),Total Multiple-User Factory Space,US Dollar (Singapore Dollar Per US Dollar),TPI % Change
0,27/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"2,852",Non-First Floor,"1,070,000",375,30 yrs from 01/03/2008,Resale,Central Region,...,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7
1,20/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"1,410",Non-First Floor,"430,000",305,30 yrs from 01/03/2008,Resale,Central Region,...,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7
2,22/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"5,350",First Floor,"3,157,000",590,99 yrs from 27/01/1984,Resale,West Region,...,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7


In [28]:
# At this point of time, my dataset has incorporated the economic variables
print(finalDF.info())
display(finalDF.head(3))

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10010 entries, 0 to 10009
Data columns (total 23 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Contract Date                               10010 non-null  object 
 1   Project Name                                9974 non-null   object 
 2   Street Name                                 10010 non-null  object 
 3   Area (sqft)                                 10010 non-null  object 
 4   Floor Level                                 10010 non-null  object 
 5   Price                                       10010 non-null  object 
 6   Unit Price ($ psf)                          10010 non-null  object 
 7   Tenure                                      10010 non-null  object 
 8   Type of Sale                                10010 non-null  object 
 9   Region                                      10010 non-null  object 
 10  Planning A

,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Price,Unit Price ($ psf),Tenure,Type of Sale,Region,...,Quarter,Month,Year,Remaining Tenure,Total Population (Number),SORA,Real GDP (in $Sm),Total Multiple-User Factory Space,US Dollar (Singapore Dollar Per US Dollar),TPI % Change
0,27/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"2,852",Non-First Floor,"1,070,000",375,30 yrs from 01/03/2008,Resale,Central Region,...,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7
1,20/12/2022,ONE COMMONWEALTH,COMMONWEALTH LANE,"1,410",Non-First Floor,"430,000",305,30 yrs from 01/03/2008,Resale,Central Region,...,2022 4Q,12,2022,16,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7
2,22/12/2022,PANTECH BUSINESS HUB,PANDAN LOOP,"5,350",First Floor,"3,157,000",590,99 yrs from 27/01/1984,Resale,West Region,...,2022 4Q,12,2022,61,5637022.0,2.5293,521934.96,790000.0,1.3789,30.7


In [29]:
# Cast the columns, Area and Price, to float datatype
finalDF['Price'] = finalDF['Price'].str.replace(',', '').astype(float)
finalDF['Area (sqft)'] = finalDF['Area (sqft)'].str.replace(',', '').astype(float)
finalDF['Unit Price ($ psf)'] = (finalDF['Price']/finalDF['Area (sqft)']).round(2)

# Concat to form the geographical locations
if 'Location' not in list(finalDF.columns): 
    finalDF['Location'] = finalDF['Region'] + '-' + finalDF['Planning Area'] + '-' + finalDF['Postal District'].astype(str) + '-' + finalDF['Postal Sector'].astype(str)        


In [30]:
# Drop remove unneeded columns & rearrrange columns
finalDF = finalDF[['Contract Date', 'Project Name', 'Street Name', 'Area (sqft)', 'Floor Level',
                   'Type of Sale', 'Region','Remaining Tenure','Total Population (Number)', 'SORA',
                   'Real GDP (in $Sm)','Total Multiple-User Factory Space','US Dollar (Singapore Dollar Per US Dollar)',
                   'TPI % Change', 'Price','Unit Price ($ psf)','Location','Planning Area', 'Postal District', 
                   'Postal Sector', 'Quarter', 'Month','Year']]

# Rename Colums for better clarity
column_mapping = {'Remaining Tenure': 'Remaining Tenure (Years)', 
                  'SORA': 'SORA (%)', 
                  'Total Multiple-User Factory Space': 'Total Multiple-User Factory Space (m2)',
                  'Total Population (Number)':'Total Population'}
finalDF.rename(columns=column_mapping, inplace=True)

# Cast columns to proper datatype
finalDF = finalDF.astype({'SORA (%)': 'float64', 
                          'US Dollar (Singapore Dollar Per US Dollar)': 'float64', 
                          'TPI % Change': 'float64'})
#Round off values
finalDF['SORA (%)'] = finalDF['SORA (%)'].round(2)
finalDF['US Dollar (Singapore Dollar Per US Dollar)'] = finalDF['US Dollar (Singapore Dollar Per US Dollar)'].round(2)
finalDF['TPI % Change'] = finalDF['TPI % Change'].round(1)

In [31]:
display(finalDF.shape)
display(finalDF.info())


(10010, 23)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10010 entries, 0 to 10009
Data columns (total 23 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Contract Date                               10010 non-null  object 
 1   Project Name                                9974 non-null   object 
 2   Street Name                                 10010 non-null  object 
 3   Area (sqft)                                 10010 non-null  float64
 4   Floor Level                                 10010 non-null  object 
 5   Type of Sale                                10010 non-null  object 
 6   Region                                      10010 non-null  object 
 7   Remaining Tenure (Years)                    10010 non-null  int64  
 8   Total Population                            10010 non-null  float64
 9   SORA (%)                                    10010 non-null  float64
 10  Real GDP (

None

Adding Extra Variables to finalDF - Added after mapping to Eco variables to save computing cost.
======================
1. Lagged psf values
2. Rental Index % Change
3. minDistToMRT 

In [32]:
print(finalDF['Project Name'].unique())
print(len(finalDF['Project Name'].unique()))

['ONE COMMONWEALTH' 'PANTECH BUSINESS HUB' 'CT HUB 2'
 'THE COMMERZE@IRVING' 'TRIVEX' 'UB POINT' 'ARK@KB' 'UBI TECHPARK'
 'ENTREPRENEUR BUSINESS CENTRE' 'EUNOS TECHNOLINK' 'EUNOS TECHPARK'
 'PREMIER @ KAKI BUKIT' 'VERTEX' 'OXLEY BIZHUB' 'UB. ONE'
 'SHUN LI INDUSTRIAL PARK' 'OXLEY BIZHUB 2' 'PAYA UBI INDUSTRIAL PARK'
 'EXCALIBUR CENTRE' 'I-LOFTS @ CHANGI' 'S9' 'FIRST CENTRE' 'MIDVIEW CITY'
 'NORTHSTAR @ AMK' 'SING INDUSTRIAL COMPLEX' 'ECO-TECH@SUNVIEW'
 'WEST CONNECT BUILDING' 'ACE@BUROH' 'PIONEER JUNCTION' 'LW TECHNOCENTRE'
 'TRADEHUB 21' 'PIONEER POINT' 'ISPACE' 'WEST STAR' 'TUAS LOT'
 'PIONEER CENTRE' 'WCEGA TOWER' 'PRESTIGE CENTRE' 'WCEGA PLAZA'
 'UNITY CENTRE' 'ENTERPRISE CENTRE' 'MANDAI CONNECTION' 'MEGA@WOODLANDS'
 'PRIMZ BIZHUB' 'WOODLANDS EAST INDUSTRIAL ESTATE' 'E9 PREMIUM'
 'PROXIMA@GAMBAS' 'WAVE9' 'NORTH LINK BUILDING' "A'POSH BIZHUB"
 'NORTH VIEW BIZHUB' 'WIN 5' 'NORTH POINT BIZHUB' 'KEWALRAM HOUSE'
 'PETRO CENTRE' 'E-CENTRE @ REDHILL' 'THE ALEXCIER' 'REDHILL FORUM'
 'CT HU

In [33]:
def addLaggedpsf(df):
    listofProject = df['Project Name'].unique()
    print('Number of Projects: {}'.format(len(listofProject)))
    
    fullDF = pd.DataFrame()
    for i in listofProject:
        projFullDF = df[df['Project Name'] == i].reset_index(drop = True) #Creates a df for each project
        projFullDF['Contract Date'] = pd.to_datetime(projFullDF['Contract Date'], dayfirst=True) # convert 'Contract Date' to datetime
        projFullDF.sort_values(by = 'Contract Date', ascending = True, inplace=True) # sort Date in ascending order
        projFullDF.reset_index(drop=True, inplace=True)
        projSubDF = projFullDF[['Contract Date','Unit Price ($ psf)']] # subset the full project DF, join back later.
        projSubDF.set_index('Contract Date', inplace = True) #set the contractDate as df index
        projSubDF = projSubDF.rolling(3, closed= "left", min_periods= 1).max() # get the lagged psf and choose the max
        projSubDF.fillna(method='bfill', inplace=True) # replace NA values with the next row's psf
        projSubDF.rename(columns={'Unit Price ($ psf)':'lagged psf'}, inplace=True)
        projSubDF.reset_index(inplace=True)
        projFullDF = projFullDF.merge(projSubDF,left_index = True, right_index=True, sort=False)
        projFullDF.rename(columns={'Contract Date_x':'Contract Date'}, inplace=True)
        fullDF = pd.concat([fullDF, projFullDF], sort=False)
    return fullDF


In [34]:
# Apply the custom function to the dataset, finalDF
laggedDF = addLaggedpsf(finalDF)
laggedDF.dropna(subset=['lagged psf'], inplace=True)
display(laggedDF.shape)
final_laggedDF = laggedDF.drop(columns=['Contract Date_y'])


Number of Projects: 144


(9969, 25)

In [35]:
display(final_laggedDF.shape)
display(final_laggedDF.info())
display(final_laggedDF[['Contract Date', 'Project Name','Unit Price ($ psf)','lagged psf']])

(9969, 24)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9969 entries, 0 to 1
Data columns (total 24 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Contract Date                               9969 non-null   datetime64[ns]
 1   Project Name                                9969 non-null   object        
 2   Street Name                                 9969 non-null   object        
 3   Area (sqft)                                 9969 non-null   float64       
 4   Floor Level                                 9969 non-null   object        
 5   Type of Sale                                9969 non-null   object        
 6   Region                                      9969 non-null   object        
 7   Remaining Tenure (Years)                    9969 non-null   int64         
 8   Total Population                            9969 non-null   float64       
 9   SORA (%)   

None

,Contract Date,Project Name,Unit Price ($ psf),lagged psf
0,2013-10-31,ONE COMMONWEALTH,562.98,562.98
1,2013-11-27,ONE COMMONWEALTH,548.91,562.98
2,2014-01-17,ONE COMMONWEALTH,552.71,562.98
3,2014-02-12,ONE COMMONWEALTH,573.35,562.98
4,2014-03-14,ONE COMMONWEALTH,541.87,573.35
...,...,...,...,...
1,2014-07-25,CRESCENDAS PRINT MEDIA HUB,499.94,500.00
0,2013-11-21,NORTHLAND INDUSTRIAL BUILDING 1,247.50,247.50
1,2014-04-21,NORTHLAND INDUSTRIAL BUILDING 1,330.71,247.50
0,2013-11-29,BRADDELL TECH,223.62,223.62


Adding the Rental Index Values
=====================
1. As requested by Clarence 28/02/2024

In [36]:
# Custom function to add Regional Rental Index % change
def mergeRentalIndex(list):
    allRentalIndexDF = pd.DataFrame()
    for i in list:
        fileName = 'Data/Rental Index of Multiple-User Factory (' + i + ').csv'
        rentalIndexDF = pd.read_csv(fileName,na_values=['NA', 'N/A','na','-'])
        rentalIndexDF.columns = ['Quarter', 'Rental Index']
        rentalIndexDF['Region'] = i
        allRentalIndexDF = pd.concat([allRentalIndexDF, rentalIndexDF])
    allRentalIndexDF.dropna(inplace=True)
    allRentalIndexDF.reset_index(drop=True, inplace=True)
    allRentalIndexDF = allRentalIndexDF[['Quarter', 'Region', 'Rental Index']]
    allRentalIndexDF['Quarter'] = allRentalIndexDF['Quarter'].apply(lambda x: x[:4] + " " + x[5] + x[4])
    return allRentalIndexDF

In [37]:
# Read the Rental Index csv file for each region and merge all of them into a single DataFrame
# Remember the Base Reference Year is 2012 Q4 
regionList = ['Central Region','East Region','North Region','North-East Region','West Region']
allRentalIndexDF = mergeRentalIndex(regionList)
allRentalIndexDF['Rental Index % change'] = allRentalIndexDF['Rental Index'] - 100


In [38]:
# Merge allRentalIndexDF with the main dataset, final_laggedDF, to map the Regional Rental Index to each property
# Reset the index for good measure
final_RICDF = final_laggedDF.merge(allRentalIndexDF ,how='inner', on=['Quarter','Region'])
final_RICDF.reset_index(drop = True, inplace=True)


In [39]:
display(final_RICDF.shape)
display(final_RICDF.info())

(9969, 26)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9969 entries, 0 to 9968
Data columns (total 26 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Contract Date                               9969 non-null   datetime64[ns]
 1   Project Name                                9969 non-null   object        
 2   Street Name                                 9969 non-null   object        
 3   Area (sqft)                                 9969 non-null   float64       
 4   Floor Level                                 9969 non-null   object        
 5   Type of Sale                                9969 non-null   object        
 6   Region                                      9969 non-null   object        
 7   Remaining Tenure (Years)                    9969 non-null   int64         
 8   Total Population                            9969 non-null   float64       
 9   SORA (%)

None

Incorporating GeoGraphical Data
=======================
1. MRT Station Coordinate taken online - Source seems to be from OneMap.gov.sg
2. Get the distinct building names
3. Reverse Geocode each property by their Project Name (ideally) or Street Name - Using OneMap.gov.sg API
4. Calculate the distance between Property and MRT Exit using the The Haversine (or great circle) distance (from SkLearn)

In [40]:
from sklearn.metrics.pairwise import haversine_distances
from math import radians
import requests
import json

# https://github.com/xkjyeah/MRT-and-LRT-Stations/blob/master/mrt_lrt.csv
mrtDF = pd.read_csv('Data\mrt_lrt.csv') #Read MRT Coords

In [41]:
# Custom Function to get the lat and long coordinates of each building, identified by the project name
# Using API from https://www.onemap.gov.sg/
def geocodeProperty(list):
    nameList,latList, longList = [], [], []
    for i in list:
        proj_name= i[0]
        nameList.append(proj_name)
        url_proj = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y".format(proj_name)
        proj_response = requests.request("GET", url_proj)
        projDict = json.loads(proj_response.text)
        
        street_name = i[1]
        url_street = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y".format(street_name)
        street_response = requests.request("GET", url_street)
        streetDict = json.loads(street_response.text)
               
        if projDict['found'] == 0:
            latList.append(streetDict['results'][0]['LATITUDE'])
            longList.append(streetDict['results'][0]['LONGITUDE'])
        else:
            latList.append(projDict['results'][0]['LATITUDE'])
            longList.append(projDict['results'][0]['LONGITUDE'])
    geoCodeDF = pd.DataFrame({'Project Name':nameList,
                              'latitude':pd.Series(latList,name = 'latitude'),
                              'longitude':pd.Series(longList,name = 'longitude')})    
    return geoCodeDF


In [43]:
# Get the unique building names from the dataset and geocode them
nameDF = final_RICDF[['Project Name','Street Name']].drop_duplicates('Project Name').reset_index(drop=True)
nameList = nameDF.values.tolist() # a Total of 138 unique building names, caution of "Woodlands east industrial estate"
display(len(nameList))

# Invoke the custome function to geocode the properties
propertyGeoCodeDF = geocodeProperty(nameList)
propertyGeoCodeDF = propertyGeoCodeDF.astype({'latitude':'float64',
                                              'longitude':'float64'})

display(propertyGeoCodeDF)
display(propertyGeoCodeDF.info())

138

,Project Name,latitude,longitude
0,ONE COMMONWEALTH,1.304428,103.796794
1,CT HUB 2,1.311588,103.863375
2,TRIVEX,1.335786,103.885510
3,UBI TECHPARK,1.326337,103.896262
4,VERTEX,1.332960,103.894116
...,...,...,...
133,PROXIMA@GAMBAS,1.441958,103.817894
134,WAVE9,1.448536,103.797449
135,NORDCOM TWO,1.443094,103.815894
136,MEGA@WOODLANDS,1.438091,103.807119


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Project Name  138 non-null    object 
 1   latitude      138 non-null    float64
 2   longitude     138 non-null    float64
dtypes: float64(2), object(1)
memory usage: 3.4+ KB


None

In [44]:
# Prepare the MRT dataset from csv file
mrtDF = mrtDF[['Name','Latitude','Longitude']]
mrtDF.rename(columns={'Name':'mrt_station',
                      'Latitude':'latitude',
                      'Longitude':'longitude'}, inplace=True)
display(mrtDF)
display(mrtDF.info())

,mrt_station,latitude,longitude
0,ADMIRALTY MRT STATION,1.440585,103.800988
1,ALJUNIED MRT STATION,1.316433,103.882906
2,ANG MO KIO MRT STATION,1.369933,103.849558
3,BARTLEY MRT STATION,1.342501,103.880178
4,BAYFRONT MRT STATION,1.281874,103.859080
...,...,...,...
188,TECK WHYE LRT STATION,1.376685,103.753712
189,TEN MILE JUNCTION LRT STATION,1.380321,103.760140
190,THANGGAM LRT STATION,1.397318,103.875635
191,TONGKANG LRT STATION,1.389348,103.885844


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   mrt_station  193 non-null    object 
 1   latitude     193 non-null    float64
 2   longitude    193 non-null    float64
dtypes: float64(2), object(1)
memory usage: 4.7+ KB


None

In [45]:
# # Computing the distance using haversine_distances
# # The Haversine (or great circle) distance is the angular distance between two points on the surface of a sphere. 
# # The first coordinate of each point is assumed to be the latitude, the second is the longitude, given in radians. 
# # The dimension of the data must be 2
# # 138 Buildings, 193 MRT & LRT station

# Use a loop structure to compute the nearest mrt distance
distList, minDistList = [], []
for i in range(propertyGeoCodeDF.shape[0]): # Outer loop to loop through all the buildings, from propertyGeoCodeDF
    propCoord = propertyGeoCodeDF.loc[i,['latitude','longitude']]
    propCoord_in_rad = [radians(_) for _ in propCoord]
    for i in range(mrtDF.shape[0]): # Inner loop to loop through all the mrt exits
        mrtCoord = mrtDF.loc[i,['latitude','longitude']]
        mrtDist_in_rad = [radians(_) for _ in mrtCoord] #Convert to radian according to docs
        result = haversine_distances([mrtDist_in_rad, propCoord_in_rad])
        resultInKM = (result * 6371)
        distList.append(resultInKM[0,1].round(5))
    minDistList.append(min(distList))
    distList = [] # Clear the list after each iteration
display(len(minDistList))

138

In [46]:
# Merge minDistToMRT with dataset with building coordinate, map back to main dataset.
minDistToMRT = pd.Series(minDistList, name = 'minDistToMRT')
propertyGeoCodeDF = pd.concat([propertyGeoCodeDF, minDistToMRT], axis = 1)
propertyGeoCodeDF['minDistToMRT'] = (propertyGeoCodeDF['minDistToMRT'] * 1000)


In [47]:
display(propertyGeoCodeDF.info())
display(propertyGeoCodeDF.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Project Name  138 non-null    object 
 1   latitude      138 non-null    float64
 2   longitude     138 non-null    float64
 3   minDistToMRT  138 non-null    float64
dtypes: float64(3), object(1)
memory usage: 4.4+ KB


None

,Project Name,latitude,longitude,minDistToMRT
0,ONE COMMONWEALTH,1.304428,103.796794,266.28
1,CT HUB 2,1.311588,103.863375,235.93
2,TRIVEX,1.335786,103.885510,333.69
3,UBI TECHPARK,1.326337,103.896262,521.75
4,VERTEX,1.332960,103.894116,657.96
5,OXLEY BIZHUB,1.332495,103.890185,349.18
6,UB. ONE,1.333512,103.891794,413.29
7,OXLEY BIZHUB 2,1.332232,103.892254,531.28
8,PAYA UBI INDUSTRIAL PARK,1.325278,103.898203,534.52
9,MIDVIEW CITY,1.358978,103.833891,519.13


In [48]:
# Merge the minDistToMRT to the main Dataset - final_RICDF
Final_Dataset = final_RICDF.merge(propertyGeoCodeDF, how = 'inner', left_on='Project Name', 
                             right_on='Project Name', sort=False)

In [49]:
display(Final_Dataset.shape)
display(Final_Dataset.info())
display(Final_Dataset.head(10))

(9969, 29)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9969 entries, 0 to 9968
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Contract Date                               9969 non-null   datetime64[ns]
 1   Project Name                                9969 non-null   object        
 2   Street Name                                 9969 non-null   object        
 3   Area (sqft)                                 9969 non-null   float64       
 4   Floor Level                                 9969 non-null   object        
 5   Type of Sale                                9969 non-null   object        
 6   Region                                      9969 non-null   object        
 7   Remaining Tenure (Years)                    9969 non-null   int64         
 8   Total Population                            9969 non-null   float64       
 9   SORA (%)

None

,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Type of Sale,Region,Remaining Tenure (Years),Total Population,SORA (%),...,Postal Sector,Quarter,Month,Year,lagged psf,Rental Index,Rental Index % change,latitude,longitude,minDistToMRT
0,2013-10-31,ONE COMMONWEALTH,COMMONWEALTH LANE,2842.0,Non-First Floor,Resale,Central Region,25,5399162.0,0.03,...,14,2013 4Q,10,2013,562.98,100.5,0.5,1.304428,103.796794,266.28
1,2013-11-27,ONE COMMONWEALTH,COMMONWEALTH LANE,1421.0,Non-First Floor,Resale,Central Region,25,5399162.0,0.05,...,14,2013 4Q,11,2013,562.98,100.5,0.5,1.304428,103.796794,266.28
2,2014-01-17,ONE COMMONWEALTH,COMMONWEALTH LANE,1679.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.05,...,14,2014 1Q,1,2014,562.98,100.8,0.8,1.304428,103.796794,266.28
3,2014-02-12,ONE COMMONWEALTH,COMMONWEALTH LANE,1636.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.15,...,14,2014 1Q,2,2014,562.98,100.8,0.8,1.304428,103.796794,266.28
4,2014-03-14,ONE COMMONWEALTH,COMMONWEALTH LANE,1421.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.21,...,14,2014 1Q,3,2014,573.35,100.8,0.8,1.304428,103.796794,266.28
5,2014-03-27,ONE COMMONWEALTH,COMMONWEALTH LANE,818.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.21,...,14,2014 1Q,3,2014,573.35,100.8,0.8,1.304428,103.796794,266.28
6,2014-04-28,ONE COMMONWEALTH,COMMONWEALTH LANE,1421.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.17,...,14,2014 2Q,4,2014,718.83,100.8,0.8,1.304428,103.796794,266.28
7,2014-07-08,ONE COMMONWEALTH,COMMONWEALTH LANE,2390.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.08,...,14,2014 3Q,7,2014,718.83,99.7,-0.3,1.304428,103.796794,266.28
8,2014-08-28,ONE COMMONWEALTH,COMMONWEALTH LANE,1195.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.18,...,14,2014 3Q,8,2014,718.83,99.7,-0.3,1.304428,103.796794,266.28
9,2014-11-21,ONE COMMONWEALTH,COMMONWEALTH LANE,1421.0,Non-First Floor,Resale,Central Region,24,5469724.0,0.20,...,14,2014 4Q,11,2014,602.51,101.1,1.1,1.304428,103.796794,266.28


In [50]:
# Rearrange Columns for better visibility
Final_Dataset = Final_Dataset[['Contract Date', 'Project Name', 'Street Name', 'Area (sqft)', 'Floor Level',
                   'Type of Sale', 'Region','Remaining Tenure (Years)','Total Population', 'SORA (%)','Real GDP (in $Sm)',
                    'Total Multiple-User Factory Space (m2)','US Dollar (Singapore Dollar Per US Dollar)',
                   'TPI % Change','lagged psf','Rental Index % change','minDistToMRT','Price','Unit Price ($ psf)',
                    'Location','Planning Area', 'Postal District','Postal Sector', 'Quarter', 'Month','Year','Rental Index',
                    'latitude','longitude']]

#Round off values
Final_Dataset['Rental Index % change'] = Final_Dataset['Rental Index % change'].round(2)
Final_Dataset['minDistToMRT'] = Final_Dataset['minDistToMRT'].round(2)


display(Final_Dataset.shape)
display(Final_Dataset.info())


(9969, 29)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9969 entries, 0 to 9968
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Contract Date                               9969 non-null   datetime64[ns]
 1   Project Name                                9969 non-null   object        
 2   Street Name                                 9969 non-null   object        
 3   Area (sqft)                                 9969 non-null   float64       
 4   Floor Level                                 9969 non-null   object        
 5   Type of Sale                                9969 non-null   object        
 6   Region                                      9969 non-null   object        
 7   Remaining Tenure (Years)                    9969 non-null   int64         
 8   Total Population                            9969 non-null   float64       
 9   SORA (%)

None

In [51]:
# Shuffle the Dataset for randomness - sklearn Shuffle
from sklearn.utils import shuffle
Final_Dataset = shuffle(Final_Dataset, random_state = 42)
Final_Dataset = Final_Dataset.reset_index(drop=True)


In [52]:
display(Final_Dataset.shape)
display(Final_Dataset.info())
display(Final_Dataset)

(9969, 29)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9969 entries, 0 to 9968
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Contract Date                               9969 non-null   datetime64[ns]
 1   Project Name                                9969 non-null   object        
 2   Street Name                                 9969 non-null   object        
 3   Area (sqft)                                 9969 non-null   float64       
 4   Floor Level                                 9969 non-null   object        
 5   Type of Sale                                9969 non-null   object        
 6   Region                                      9969 non-null   object        
 7   Remaining Tenure (Years)                    9969 non-null   int64         
 8   Total Population                            9969 non-null   float64       
 9   SORA (%)

None

,Contract Date,Project Name,Street Name,Area (sqft),Floor Level,Type of Sale,Region,Remaining Tenure (Years),Total Population,SORA (%),...,Location,Planning Area,Postal District,Postal Sector,Quarter,Month,Year,Rental Index,latitude,longitude
0,2015-01-13,VERTEX,UBI AVENUE 3,1733.0,Non-First Floor,Resale,Central Region,52,5535002.0,0.39,...,Central Region-Geylang-14-40,Geylang,14,40,2015 1Q,1,2015,102.4,1.332960,103.894116
1,2022-06-02,E-CENTRE @ REDHILL,JALAN BUKIT MERAH,1981.0,Non-First Floor,Resale,Central Region,39,5637022.0,1.66,...,Central Region-Bukit Merah-3-15,Bukit Merah,3,15,2022 2Q,6,2022,93.0,1.285034,103.811609
2,2017-08-21,E9 PREMIUM,WOODLANDS INDUSTRIAL PARK E9,2454.0,First Floor,Resale,North Region,26,5612253.0,0.52,...,North Region-Woodlands-27-75,Woodlands,27,75,2017 3Q,8,2017,91.0,1.449236,103.798954
3,2014-09-30,CT HUB 2,LAVENDER STREET,1109.0,Non-First Floor,New Sale,Central Region,61,5469724.0,0.08,...,Central Region-Kallang-12-33,Kallang,12,33,2014 3Q,9,2014,99.7,1.311588,103.863375
4,2021-07-12,UBI TECHPARK,UBI CRESCENT,1851.0,Non-First Floor,Resale,Central Region,36,5453566.0,0.27,...,Central Region-Geylang-14-40,Geylang,14,40,2021 3Q,7,2021,90.9,1.326337,103.896262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9964,2019-10-11,ENTERPRISE ONE,KAKI BUKIT ROAD 1,2153.0,Non-First Floor,Resale,East Region,37,5703569.0,1.64,...,East Region-Bedok-14-41,Bedok,14,41,2019 4Q,10,2019,91.0,1.332748,103.902616
9965,2022-04-19,EUNOS TECHNOLINK,KAKI BUKIT ROAD 1,3089.0,Non-First Floor,Resale,East Region,34,5637022.0,0.68,...,East Region-Bedok-14-41,Bedok,14,41,2022 2Q,4,2022,93.0,1.334257,103.901404
9966,2021-08-06,PREMIER @ KAKI BUKIT,KAKI BUKIT AVENUE 4,1098.0,Non-First Floor,Resale,East Region,49,5453566.0,0.22,...,East Region-Bedok-14-41,Bedok,14,41,2021 3Q,8,2021,90.1,1.338521,103.906567
9967,2015-09-18,UB. ONE,UBI AVENUE 4,926.0,Non-First Floor,Resale,Central Region,53,5535002.0,0.22,...,Central Region-Geylang-14-40,Geylang,14,40,2015 3Q,9,2015,101.9,1.333512,103.891794


In [53]:
# # Remember this is my final dataset, with all the outliers not removed. 
# # I should probably use this for PowerBI for a better picture for SNRE.
# # Export to .csv
# Final_Dataset.to_csv('SNRE_Dataset_2024_Outliers.csv', index=False)

In [54]:
# display(Final_DataDF.info())